# Notebook 04 — Formulación y validación de hipótesis (Fases 2 y 6)

**Qué hace este notebook:** convierte las preguntas del notebook 01 en **hipótesis contrastables**, verifica los supuestos de cada prueba, ejecuta los tests y decide con el **valor p** (α = 0,05), reportando también el **tamaño del efecto** (un p muy pequeño con 40.000 pedidos no significa que el efecto importe en la práctica).

**Regla de coherencia:** cada hipótesis se muestra en el video y en el dashboard con *su propio* gráfico. (En la entrega anterior se proyectó una figura que no correspondía a las hipótesis; aquí se evita.)

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Rutas del proyecto (los notebooks viven en /notebooks)
RAIZ = Path('..').resolve()
RAW = RAIZ / 'data' / 'raw' / 'online_retail_II.csv'
PROC = RAIZ / 'data' / 'processed'
FIG = RAIZ / 'docs' / 'figuras'
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Estilo comun a todos los graficos del proyecto
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, NARANJA, GRIS, VERDE = '#1F4E79', '#E8833A', '#8C8C8C', '#3B8E5A'
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Hipótesis

Nivel de significancia: **α = 0,05** para todas. Como son varias pruebas, al final se aplica la corrección de **Holm** para controlar el error tipo I acumulado.

---
### H1 — Estacionalidad (responde P1)
- **H0:** los ingresos semanales en temporada alta (septiembre–noviembre) son iguales a los del resto del año.
- **H1:** los ingresos semanales en temporada alta son **mayores**.
- **Variables:** `fecha` (semana), `importe` (ventas brutas). **Prueba:** Mann-Whitney U unilateral (y t de Welch sobre log como verificación).
- **Por qué importa:** si es cierto, el inventario y el marketing deben concentrarse antes de septiembre. **KPI:** Ingresos netos.

### H2 — Tamaño del pedido por mercado (P5)
- **H0:** el valor del pedido es igual en clientes del Reino Unido y del Exterior.
- **H1:** el valor del pedido **difiere** entre ambos mercados.
- **Variables:** `importe` del pedido, `pais_grupo`. **Prueba:** Mann-Whitney U bilateral (+ t de Welch sobre log).
- **Por qué importa:** decide si vale la pena una estrategia comercial distinta para exportación. **KPI:** Ticket promedio.

### H3 — Devoluciones y mercado (P4, P5)
- **H0:** la proporción de facturas de devolución es la misma en el Reino Unido y en el Exterior (independencia).
- **H1:** la proporción **difiere** entre mercados.
- **Variables:** `tipo` de factura (venta/devolución), `pais_grupo`. **Prueba:** Chi-cuadrado de independencia + V de Cramér.
- **Por qué importa:** si el Exterior devuelve más, hay un problema logístico o de expectativas concentrado ahí. **KPI:** Tasa de devolución.

### H4 — Valor del primer pedido y recompra (P3)
- **H0:** el valor del primer pedido es igual entre quienes recompran en 90 días y quienes no.
- **H1:** quienes recompran en 90 días hicieron un primer pedido de **mayor** valor.
- **Variables:** `importe` del primer pedido, `recompra_90d`. **Prueba:** Mann-Whitney U unilateral + regresión logística (para controlar por otras variables).
- **Por qué importa:** permite identificar desde el primer pedido a los clientes con más potencial. **KPI:** Tasa de recompra a 90 días.

### H5 — Captación en temporada alta y recompra (P1, P3)
- **H0:** la tasa de recompra a 90 días es igual para clientes captados en temporada alta y en el resto del año.
- **H1:** la tasa de recompra **difiere**.
- **Variables:** mes de la primera compra, `recompra_90d`. **Prueba:** Chi-cuadrado + diferencia de proporciones con IC 95 %.
- **Por qué importa:** la temporada alta atrae mucha demanda, pero ¿trae clientes que se quedan? Define dónde invertir en retención. **KPI:** Tasa de recompra.

### H6 — Mercado y recompra (P3, P5)
- **H0:** la tasa de recompra a 90 días es igual en el Reino Unido y en el Exterior.
- **H1:** la tasa de recompra **difiere**.
- **Variables:** `pais_grupo`, `recompra_90d`. **Prueba:** Chi-cuadrado + diferencia de proporciones con IC 95 %.
- **Por qué importa:** si fuera igual, no hace falta un programa de retención específico para exportación.

In [2]:
hipotesis = pd.DataFrame([
    ('H1', 'Ingresos semanales: temporada alta > resto', 'Mann-Whitney U (unilateral)', 'Ingresos netos',           'P1'),
    ('H2', 'Valor del pedido: Reino Unido != Exterior',  'Mann-Whitney U (bilateral)', 'Ticket promedio',          'P5'),
    ('H3', 'Tasa de devolucion: Reino Unido != Exterior', 'Chi-cuadrado + V de Cramer', 'Tasa de devolucion',      'P4, P5'),
    ('H4', 'Primer pedido: recompra > no recompra',        'Mann-Whitney U + logistica', 'Tasa de recompra 90d',    'P3'),
    ('H5', 'Recompra: captados en temporada alta != resto', 'Chi-cuadrado + IC',          'Tasa de recompra 90d',    'P1, P3'),
    ('H6', 'Recompra: Reino Unido != Exterior',            'Chi-cuadrado + IC',          'Tasa de recompra 90d',    'P3, P5'),
], columns=['id', 'hipotesis', 'prueba', 'kpi relacionado', 'pregunta'])
hipotesis

,id,hipotesis,prueba,kpi relacionado,pregunta
0,H1,Ingresos semanales: temporada alta > resto,Mann-Whitney U (unilateral),Ingresos netos,P1
1,H2,Valor del pedido: Reino Unido != Exterior,Mann-Whitney U (bilateral),Ticket promedio,P5
2,H3,Tasa de devolucion: Reino Unido != Exterior,Chi-cuadrado + V de Cramer,Tasa de devolucion,"P4, P5"
3,H4,Primer pedido: recompra > no recompra,Mann-Whitney U + logistica,Tasa de recompra 90d,P3
4,H5,Recompra: captados en temporada alta != resto,Chi-cuadrado + IC,Tasa de recompra 90d,"P1, P3"
5,H6,Recompra: Reino Unido != Exterior,Chi-cuadrado + IC,Tasa de recompra 90d,"P3, P5"
